# LangChain — Подробный разбор

## Что такое LangChain?

**LangChain** — это фреймворк для создания приложений с LLM (Large Language Models).

### Зачем нужен, если можно просто вызвать API?

```python
# Без LangChain — просто вызов API
response = openai.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Привет"}]
)
```

Это работает для простых случаев. Но когда нужно:
- Цепочки вызовов (результат одного → вход другого)
- Память диалога
- Вызов внешних функций (tools)
- RAG (поиск в документах)
- Агенты (LLM сама решает что делать)

...код становится сложным. LangChain даёт готовые абстракции.

### Аналогия с Java:

| LangChain | Java аналогия |
|-----------|---------------|
| LangChain | Spring Framework |
| Chain | Pipeline / Chain of Responsibility |
| Prompt Template | String.format() / MessageFormat |
| Memory | HttpSession / кэш |
| Tools | Strategy pattern / внешние сервисы |
| Agent | Finite State Machine |

In [1]:
# Загружаем API ключи из .env файла
# .env файл содержит: DEEPSEEK_API_KEY=sk-xxx
from dotenv import load_dotenv
import os

load_dotenv("../../.env")  # путь относительно ноутбука
print("API ключ загружен")
print(f"Ключ начинается с: {os.getenv('DEEPSEEK_API_KEY')[:10]}...")

API ключ загружен
Ключ начинается с: sk-***...


## 1. Chat Model — обёртка над LLM API

### Что это?

`ChatOpenAI` — это класс, который оборачивает вызов API в удобный интерфейс.

### Почему DeepSeek работает через ChatOpenAI?

DeepSeek API **совместим с OpenAI API** — те же endpoints, тот же формат запросов.
Просто меняем `base_url` и `api_key`.

```
OpenAI:   https://api.openai.com/v1
DeepSeek: https://api.deepseek.com
```

### Параметры модели:

| Параметр | Что делает | Значение |
|----------|------------|----------|
| `temperature` | Креативность (0=точно, 1=креативно) | 0.7 |
| `max_tokens` | Максимум токенов в ответе | 1000 |
| `top_p` | Nucleus sampling | 0.9 |
| `model` | Название модели | "deepseek-chat" |

In [2]:
from langchain_openai import ChatOpenAI

# Создаём объект модели
# Это НЕ делает запрос к API — просто конфигурация
llm = ChatOpenAI(
    model="deepseek-chat",                    # название модели
    api_key=os.getenv("DEEPSEEK_API_KEY"),   # ключ из .env
    base_url="https://api.deepseek.com",     # URL API (не OpenAI!)
    temperature=0.7                           # креативность
)

print(f"Модель: {llm.model_name}")
print(f"Temperature: {llm.temperature}")

C:\Users\Ruslan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Модель: deepseek-chat
Temperature: 0.7


In [3]:
# Простой вызов — метод invoke()
# Это СИНХРОННЫЙ вызов — ждём пока API ответит

response = llm.invoke("Привет! Представься кратко, в 2 предложениях.")

# response — это объект AIMessage, не строка!
print(f"Тип ответа: {type(response)}")
print(f"\nТекст ответа: {response.content}")

Тип ответа: <class 'langchain_core.messages.ai.AIMessage'>

Текст ответа: Привет! Я DeepSeek, умный AI-ассистент, созданный компанией DeepSeek. Готов помочь с решением задач, ответами на вопросы и поддержкой в различных темах.


### Типы сообщений в LangChain

LangChain использует типизированные сообщения:

```python
from langchain_core.messages import (
    SystemMessage,   # инструкции для модели (role: system)
    HumanMessage,    # сообщение пользователя (role: user)
    AIMessage,       # ответ модели (role: assistant)
    ToolMessage      # результат вызова инструмента
)
```

Это типизированный аналог:
```json
{"role": "system", "content": "..."}
{"role": "user", "content": "..."}
{"role": "assistant", "content": "..."}
```

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

# Вызов с несколькими сообщениями
messages = [
    SystemMessage(content="Ты пират. Отвечай как пират, используя 'Аррр!' и морские термины."),
    HumanMessage(content="Как дела?")
]

response = llm.invoke(messages)
print(response.content)

Аррр! Дела идут как по морю в шторм — то вверх, то вниз, но паруса подняты и курс держим твёрдо! А у тебя как, земляк?


## 2. Prompt Templates — шаблоны промптов

### Зачем нужны?

Вместо конкатенации строк:
```python
# Плохо — сложно читать, легко ошибиться
prompt = "Ты " + role + ". Ответь на вопрос: " + question
```

Используем шаблоны:
```python
# Хорошо — чисто и переиспользуемо
prompt = "Ты {role}. Ответь на вопрос: {question}"
```

### Аналогия с Java:

```java
// Java String.format()
String.format("Ты %s. Ответь на вопрос: %s", role, question);

// Или MessageFormat
MessageFormat.format("Ты {0}. Ответь на вопрос: {1}", role, question);
```

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Создаём шаблон
# Переменные в {фигурных скобках}
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты {role}. Отвечай кратко на русском, максимум 3 предложения."),
    ("user", "{question}")
])

# Посмотрим что внутри
print("Переменные в шаблоне:", prompt.input_variables)

Переменные в шаблоне: ['question', 'role']


In [6]:
# Форматируем шаблон с конкретными значениями
# Это ещё НЕ вызывает LLM — просто подставляет переменные

formatted_messages = prompt.format_messages(
    role="опытный Python разработчик с 10 годами опыта",
    question="Что лучше: Django или FastAPI?"
)

print("Сформированные сообщения:")
for msg in formatted_messages:
    print(f"\n[{msg.type}]: {msg.content}")

Сформированные сообщения:

[system]: Ты опытный Python разработчик с 10 годами опыта. Отвечай кратко на русском, максимум 3 предложения.

[human]: Что лучше: Django или FastAPI?


In [7]:
# Теперь отправляем в LLM
response = llm.invoke(formatted_messages)
print("Ответ LLM:")
print(response.content)

Ответ LLM:
Выбор зависит от задачи. Django — это полноценный фреймворк с ORM, админкой и многими встроенными компонентами, идеально подходит для монолитных приложений. FastAPI — современный микрофреймворк для быстрого создания API с асинхронной поддержкой и автоматической документацией. Для сложных веб-приложений с большим количеством функций выбирайте Django, а для высокопроизводительных API или микросервисов — FastAPI.


## 3. Chains — цепочки вызовов (LCEL)

### Что такое Chain?

**Chain** — это последовательность операций, где выход одной → вход следующей.

```
Input → [Prompt] → [LLM] → [Parser] → Output
```

### LCEL — LangChain Expression Language

Новый синтаксис для создания chains через оператор `|` (pipe):

```python
chain = prompt | llm | parser
```

### Аналогия с Java:

```java
// Java Streams — похожий pipe синтаксис
result = data.stream()
    .map(formatter::format)  // prompt
    .map(llm::call)          // llm
    .map(parser::parse)      // parser
    .findFirst();
```

In [8]:
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser — извлекает текст из AIMessage
# AIMessage.content → str

parser = StrOutputParser()

# Создаём chain: prompt → llm → parser
# Оператор | соединяет компоненты
chain = prompt | llm | parser

print(f"Тип chain: {type(chain).__name__}")

Тип chain: RunnableSequence


In [9]:
# Вызываем chain — передаём словарь с переменными
# Chain сам:
# 1. Подставит переменные в prompt
# 2. Отправит в LLM
# 3. Извлечёт текст из ответа

result = chain.invoke({
    "role": "senior Java разработчик",
    "question": "Зачем нужен Optional в Java 8?"
})

# Теперь result — это строка, не AIMessage!
print(f"Тип результата: {type(result).__name__}")
print(f"\nРезультат:\n{result}")

Тип результата: str

Результат:
Optional в Java 8 вводится для явного представления возможного отсутствия значения, заменяя null. Это помогает избежать NullPointerException и делает код более читаемым, заставляя явно обрабатывать случай пустого значения.


In [10]:
# Пример: цепочка переводов
# Переведём текст: Русский → Английский → Японский

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Переведи текст на {language}. Только перевод, без пояснений."),
    ("user", "{text}")
])

translate_chain = translate_prompt | llm | StrOutputParser()

# Шаг 1: Русский → Английский
original = "Привет! Как дела? Сегодня отличная погода."
english = translate_chain.invoke({"language": "английский", "text": original})

# Шаг 2: Английский → Японский
japanese = translate_chain.invoke({"language": "японский", "text": english})

print(f"Оригинал (RU): {original}")
print(f"Английский:    {english}")
print(f"Японский:      {japanese}")

Оригинал (RU): Привет! Как дела? Сегодня отличная погода.
Английский:    Hello! How are you? The weather is great today.
Японский:      こんにちは！お元気ですか？今日は天気がとてもいいですね。


## 4. Sequential Chains — последовательные цепочки

### Когда нужно?

Когда результат одного LLM вызова нужен для следующего:

```
Задача → [Генерация идеи] → идея → [Критика] → критика → [Улучшение] → результат
```

### Паттерн "Генерация → Критика → Улучшение"

Это популярный паттерн для улучшения качества:
1. LLM генерирует первый вариант
2. LLM (или другая) критикует
3. LLM улучшает с учётом критики

Похоже на **code review** в разработке!

In [11]:
# Chain 1: Генерация идеи
idea_prompt = ChatPromptTemplate.from_messages([
    ("system", "Придумай одну бизнес-идею для {topic}. Кратко, 2-3 предложения."),
    ("user", "Тема: {topic}")
])
idea_chain = idea_prompt | llm | StrOutputParser()

# Chain 2: Критика идеи
critic_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты строгий бизнес-критик. Найди 2-3 слабых места в идее. Будь конкретным."),
    ("user", "Идея: {idea}")
])
critic_chain = critic_prompt | llm | StrOutputParser()

# Chain 3: Улучшение с учётом критики
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", "Улучши бизнес-идею, учитывая критику. Предложи решения для каждого слабого места."),
    ("user", "Исходная идея: {idea}\n\nКритика: {criticism}")
])
improve_chain = improve_prompt | llm | StrOutputParser()

print("3 chain'а созданы")

3 chain'а созданы


In [12]:
# Запускаем полный пайплайн
def run_idea_pipeline(topic: str):
    """Генерация → Критика → Улучшение"""
    
    print(f"Тема: {topic}")
    print("=" * 60)
    
    # Шаг 1: Генерация
    idea = idea_chain.invoke({"topic": topic})
    print(f"\n[1] ИДЕЯ:\n{idea}")
    
    # Шаг 2: Критика
    criticism = critic_chain.invoke({"idea": idea})
    print(f"\n[2] КРИТИКА:\n{criticism}")
    
    # Шаг 3: Улучшение
    improved = improve_chain.invoke({"idea": idea, "criticism": criticism})
    print(f"\n[3] УЛУЧШЕННАЯ ИДЕЯ:\n{improved}")
    
    return improved

# Тест
result = run_idea_pipeline("мобильное приложение для фитнеса")

Тема: мобильное приложение для фитнеса



[1] ИДЕЯ:
**Fitness Quest** — приложение, превращающее тренировки в увлекательную RPG-игру. Пользователи создают персонажа, выполняют «квесты» (реальные упражнения), зарабатывают очки опыта и улучшают характеристики, соревнуясь с друзьями. Это мотивирует регулярно заниматься спортом через геймификацию и социализацию.



[2] КРИТИКА:
Критический анализ идеи **Fitness Quest**:

1. **Проблема долгосрочной вовлеченности**: Геймификация часто теряет новизну через 3-6 месяцев. После достижения максимального уровня персонажа или прохождения основных "квестов" мотивация резко падает. Пользователи могут устать от однотипных механик, а постоянное добавление нового контента требует значительных ресурсов разработчиков.

2. **Сложность верификации выполнения упражнений**: Приложение столкнется с проблемой честности пользователей. Как система отличит реальное выполнение 20 приседаний от простого нажатия кнопки? Использование датчиков смартфона (акселерометр, гироскоп) дает высокую погрешность, а требование дополнительного оборудования (фитнес-трекеры) повышает порог входа и снижает доступность.

3. **Риск превращения в "цифровую рутину"**: Со временем даже игровые механики могут восприниматься как обязательная работа, особенно если система наград станет предсказуемой. Соревнование с друзьями может демотивировать м


[3] УЛУЧШЕННАЯ ИДЕЯ:
Отличная и конструктивная критика. Она затрагивает ключевые проблемы, с которыми сталкиваются многие геймифицированные приложения. Вот улучшенная версия идеи **Fitness Quest** с решениями для каждого слабого места, превращающая их в возможности.

### **Улучшенная идея: Fitness Quest 2.0 — Эволюция Персонажа через Реальный Прогресс**

**Основной принцип:** Это не просто игра, где тренировки — это мини-игры. Это **зеркало вашего реального фитнес-пути**, обернутое в захватывающую RPG-оболочку, которая адаптируется и растет вместе с пользователем.

---

### **Решения для критических замечаний:**

#### **1. Проблема долгосрочной вовлеченности: От "Линейной игры" к "Живому миру"**

*   **Решение А: Бесконечная прогрессия и "Эволюция" вместо "Макс. уровня".**
    *   **Система Престижа/Реинкарнации:** После достижения "максимального" уровня персонаж может "эволюционировать" — открывается новая ветка развития (напр., из "Воина" в "Мастера Стихий" или "Техноманта"), сбрасы

## 5. Output Parsers — структурированный вывод

### Проблема

LLM возвращает **текст**. Но часто нужны структурированные данные:
- JSON
- Список
- Объект с полями

### Решение: Output Parsers

1. Говорим LLM в каком формате отвечать (в промпте)
2. Парсим ответ в нужный тип

### Аналогия с Java:

```java
// Получаем JSON строку от API
String json = llm.call(prompt);

// Парсим в объект (Jackson ObjectMapper)
Movie movie = objectMapper.readValue(json, Movie.class);
```

In [13]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Определяем схему данных через Pydantic
# Это как Java DTO/Record
class Movie(BaseModel):
    """Информация о фильме."""
    title: str = Field(description="Название фильма")
    year: int = Field(description="Год выхода")
    genre: str = Field(description="Жанр фильма")
    rating: float = Field(description="Рейтинг от 1 до 10")

# Создаём parser
parser = JsonOutputParser(pydantic_object=Movie)

# Смотрим какие инструкции parser даёт для LLM
print("Инструкции для LLM:")
print(parser.get_format_instructions())

Инструкции для LLM:
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdo

In [14]:
# Создаём промпт с инструкциями по формату
movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "Придумай вымышленный фильм и верни данные в JSON формате.\n{format_instructions}"),
    ("user", "Жанр: {genre}")
])

# .partial() — подставляем часть переменных заранее
movie_prompt = movie_prompt.partial(
    format_instructions=parser.get_format_instructions()
)

# Собираем chain
movie_chain = movie_prompt | llm | parser

# Генерируем фильм
result = movie_chain.invoke({"genre": "научная фантастика"})

print(f"Тип результата: {type(result).__name__}")
print(f"\nРезультат:")
for key, value in result.items():
    print(f"  {key}: {value}")

Тип результата: dict

Результат:
  title: Эхо хроноса
  year: 2077
  genre: научная фантастика
  rating: 8.7


## 6. Memory — память диалога

### Проблема

LLM **не помнит** предыдущие сообщения. Каждый вызов — с чистого листа.

```python
llm.invoke("Меня зовут Руслан")  # OK
llm.invoke("Как меня зовут?")    # "Не знаю" — нет контекста!
```

### Решение: передавать историю в каждом запросе

```python
messages = [
    HumanMessage("Меня зовут Руслан"),
    AIMessage("Приятно познакомиться!"),
    HumanMessage("Как меня зовут?")  # теперь LLM видит контекст
]
```

### Memory в LangChain

Автоматически сохраняет и добавляет историю в каждый запрос.

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# Создаём хранилище истории (в памяти)
# В продакшне: Redis, PostgreSQL, файл
history = InMemoryChatMessageHistory()

def chat_with_memory(user_message: str) -> str:
    """
    Чат с памятью — помнит предыдущие сообщения.
    """
    # 1. Добавляем сообщение пользователя в историю
    history.add_user_message(user_message)
    
    # 2. Формируем промпт с полной историей
    messages = [
        ("system", "Ты дружелюбный ассистент. Помни контекст разговора.")
    ]
    
    # Добавляем всю историю
    for msg in history.messages:
        role = "human" if isinstance(msg, HumanMessage) else "assistant"
        messages.append((role, msg.content))
    
    prompt = ChatPromptTemplate.from_messages(messages)
    
    # 3. Вызываем LLM
    response = llm.invoke(prompt.format_messages())
    
    # 4. Сохраняем ответ в историю
    history.add_ai_message(response.content)
    
    return response.content

print("Функция чата с памятью готова")

Функция чата с памятью готова


In [16]:
# Тестируем память
print("User: Привет! Меня зовут Руслан.")
print(f"AI: {chat_with_memory('Привет! Меня зовут Руслан.')}")
print()

print("User: Я работаю Java разработчиком в Сбере.")
print(f"AI: {chat_with_memory('Я работаю Java разработчиком в Сбере.')}")
print()

print("User: Как меня зовут и кем я работаю?")
print(f"AI: {chat_with_memory('Как меня зовут и кем я работаю?')}")

User: Привет! Меня зовут Руслан.


AI: Привет, Руслан! Очень приятно познакомиться. Чем могу помочь сегодня?

User: Я работаю Java разработчиком в Сбере.


AI: Привет, Руслан! Здравствуйте, коллега (в каком-то смысле, в IT-сфере 😊). Работа в Сбере — это интересно и ответственно. У вас там наверняка масштабные проекты.

Как Java-разработчик, вы наверняка работаете с высоконагруженными системами. На каких проектах или технологическом стеке специализируетесь? (Spring Boot, микросервисы, Kafka, Big Data что-то из этого?) Или, может, есть какой-то интересный рабочий вопрос, который хотелось бы обсудить?

User: Как меня зовут и кем я работаю?


AI: Вас зовут Руслан, и вы работаете Java-разработчиком в Сбере. 😊  
Хорошо помню детали из нашего разговора!


## 7. Tools — инструменты для LLM

### Что это?

**Tools** — это функции, которые LLM может вызывать для получения данных или выполнения действий.

LLM **сама решает**:
- Нужен ли tool?
- Какой tool вызвать?
- С какими параметрами?

### Примеры tools:

- `get_weather(city)` — получить погоду
- `calculate(expression)` — вычислить выражение
- `search_database(query)` — поиск в БД
- `send_email(to, subject, body)` — отправить email

### Аналогия с Java:

```java
// Tools — это как Strategy pattern
interface Tool {
    String getName();
    String getDescription();  // LLM читает это!
    String execute(Map<String, Object> args);
}
```

In [17]:
from langchain_core.tools import tool
import random
from datetime import datetime

# Декоратор @tool превращает функцию в Tool
# ВАЖНО: Docstring становится описанием для LLM!

@tool
def get_weather(city: str) -> str:
    """
    Получить текущую погоду в указанном городе.
    Используй когда пользователь спрашивает о погоде.
    
    Args:
        city: Название города
    """
    # В реальности здесь был бы API запрос
    weather_data = {
        "москва": {"temp": -5, "condition": "снег"},
        "сочи": {"temp": 12, "condition": "солнечно"},
    }
    data = weather_data.get(city.lower(), {"temp": random.randint(-10, 25), "condition": "облачно"})
    return f"Погода в {city}: {data['temp']}°C, {data['condition']}"

@tool
def calculate(expression: str) -> str:
    """
    Вычислить математическое выражение.
    Используй для любых математических расчётов.
    
    Args:
        expression: Выражение, например "2 + 2" или "15 * 37"
    """
    try:
        result = eval(expression)  # В продакшне используй безопасный парсер!
        return f"{expression} = {result}"
    except:
        return "Ошибка вычисления"

@tool
def get_current_time() -> str:
    """Получить текущее время. Используй когда спрашивают который час."""
    return f"Текущее время: {datetime.now().strftime('%H:%M:%S')}"

tools = [get_weather, calculate, get_current_time]

print("Tools созданы:")
for t in tools:
    print(f"  - {t.name}")

Tools созданы:
  - get_weather
  - calculate
  - get_current_time


In [18]:
# Привязываем tools к LLM
# Теперь LLM знает какие tools доступны

llm_with_tools = llm.bind_tools(tools)

# Тест: LLM решает какой tool нужен
response = llm_with_tools.invoke("Какая погода в Сочи?")

print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: Я проверю текущую погоду в Сочи для вас.
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Сочи'}, 'id': 'call_00_dXzVWaegiS55W4MrUiGv3LPE', 'type': 'tool_call'}]


### Как работает Tool Calling?

```
1. User: "Какая погода в Москве?"
   ↓
2. LLM анализирует вопрос и доступные tools
   ↓
3. LLM возвращает: "Хочу вызвать get_weather(city='Москва')"
   (это tool_call — НЕ результат!)
   ↓
4. МЫ выполняем функцию: get_weather("Москва") → "Погода: -5°C"
   ↓
5. Передаём результат обратно в LLM
   ↓
6. LLM формирует финальный ответ: "В Москве -5°C и снег"
```

In [19]:
from langchain_core.messages import ToolMessage

def ask_with_tools(question: str) -> str:
    """
    Полный цикл: вопрос → tool call → выполнение → ответ
    """
    print(f"Q: {question}")
    
    # Шаг 1: Спрашиваем LLM
    messages = [("user", question)]
    response = llm_with_tools.invoke(messages)
    
    # Если LLM не хочет вызывать tool
    if not response.tool_calls:
        return response.content
    
    # Шаг 2: Выполняем tool calls
    messages.append(response)
    
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"   Tool: {tool_name}({tool_args})")
        
        # Находим и вызываем tool
        tool_fn = {t.name: t for t in tools}[tool_name]
        result = tool_fn.invoke(tool_args)
        
        print(f"   Result: {result}")
        
        messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))
    
    # Шаг 3: LLM формирует финальный ответ
    final_response = llm_with_tools.invoke(messages)
    return final_response.content

print("Функция ask_with_tools готова")

Функция ask_with_tools готова


In [20]:
# Тестируем
questions = [
    "Какая погода в Москве?",
    "Сколько будет 157 * 23?",
    "Который сейчас час?",
]

for q in questions:
    print("\n" + "="*50)
    answer = ask_with_tools(q)
    print(f"A: {answer}")


Q: Какая погода в Москве?


   Tool: get_weather({'city': 'Москва'})
   Result: Погода в Москва: -5°C, снег


A: В Москве сейчас -5°C и идет снег.

Q: Сколько будет 157 * 23?


   Tool: calculate({'expression': '157 * 23'})
   Result: 157 * 23 = 3611


A: 157 × 23 = 3611

Q: Который сейчас час?


   Tool: get_current_time({})
   Result: Текущее время: 23:02:49


A: Сейчас 23:02:49 (11 часов 2 минуты 49 секунд вечера).


## 8. RAG с LangChain

### Что такое RAG?

**RAG = Retrieval Augmented Generation**

1. Находим релевантные документы (Retrieval)
2. Добавляем их в промпт (Augmented)
3. LLM генерирует ответ на основе документов (Generation)

### LangChain упрощает RAG:

```python
# Одна строка вместо 20
chain = retriever | prompt | llm
```

In [21]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough

# Документы для RAG
documents = [
    "Python создан Гвидо ван Россумом в 1991 году.",
    "Java создан Джеймсом Гослингом в Sun Microsystems в 1995 году.",
    "JavaScript создан Бренданом Эйхом в Netscape в 1995 году.",
    "Go создан в Google в 2009 году.",
    "Rust создан в Mozilla в 2010 году.",
]

# Создаём embeddings и vectorstore
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = Chroma.from_texts(documents, embeddings)

# Retriever — ищет похожие документы
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Vectorstore создан с {len(documents)} документами")

C:\Users\Ruslan\AppData\Local\Temp\ipykernel_13884\176661340.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/199 [00:00<00:00, 758.88it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|          | 2/199 [00:00<00:00, 860.28it/s, Materializing param=embeddings.LayerNorm.weight]

...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 597.38it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 597.38it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 585.81it/s, Materializing param=pooler.dense.weight]


BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore создан с 5 документами


In [22]:
# RAG Chain
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Отвечай ТОЛЬКО на основе контекста. Если нет информации — скажи что не знаешь."),
    ("user", "Контекст:\n{context}\n\nВопрос: {question}")
])

def format_docs(docs):
    """Превращает список документов в строку."""
    return "\n".join(doc.page_content for doc in docs)

# LCEL chain для RAG
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain создан")

RAG chain создан


In [23]:
# Тестируем RAG
questions = [
    "Кто создал Python?",
    "Какой язык создан в Mozilla?",
    "Кто президент России?"  # нет в контексте
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")


Q: Кто создал Python?


A: Согласно контексту, Python создал Гвидо ван Россум.

Q: Какой язык создан в Mozilla?


A: На основе предоставленного контекста, язык, созданный в Mozilla, — это Rust, который был создан в 2010 году.

Q: Кто президент России?


A: На основе предоставленного контекста нет информации о президенте России.


## Итоги

### Что изучили:

| Концепция | Описание | Java аналогия |
|-----------|----------|---------------|
| **Chat Model** | Обёртка над LLM API | HTTP Client |
| **Prompt Template** | Шаблон с переменными | String.format() |
| **Chain (LCEL)** | Цепочка операций `a \| b \| c` | Stream API |
| **Output Parser** | Парсинг JSON/структур | Jackson ObjectMapper |
| **Memory** | История диалога | HttpSession |
| **Tools** | Функции для LLM | Strategy pattern |
| **RAG** | Поиск + генерация | Search + Template |

### Что дальше:

- **Agents** — LLM сама планирует и выполняет задачи
- **LangGraph** — граф состояний для сложных workflow
- **Streaming** — потоковый вывод ответа